# PyTorch Tensors: Tutorial 1
### From Vectorized Operations to Deep Learning Foundations


## Learning Objectives

By the end of this session, you will be able to:
1. Create and manipulate PyTorch tensors
2. Explain *why* vectorized tensor operations outperform Python loops
3. Empirically measure and compare execution times using `timeit`
4. Implement and benchmark matrix multiplication using loops vs. tensor operations
5. Understand how these building blocks connect directly to DNNs, CNNs, and other deep learning models

---

#  TUTORIAL

## Why PyTorch? Why Tensors?

Regardless of the architecture's name say DNNs, CNNs, RNNs, or Transformers, the real engine running underneath every DNN, CNN, or Transformer are  muti-dimentional array called tensor arithmetic — matrix multiplications and element-wise ops, repeated many times.

- A **scalar** is a 0-D tensor
- A **vector** is a 1-D tensor
- A **matrix** is a 2-D tensor
- An **image batch** (batch × channels × height × width) is a 4-D tensor

PyTorch tensors are similar to NumPy arrays, but with two superpowers:
1. They can run on **GPUs** for massive parallel speedups
2. They support **autograd** — automatic differentiation, which is how neural networks learn (via backpropagation)

Install (if needed): `pip install torch`


In [14]:
# !pip show torch
import sys
!{sys.executable} -m pip install torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Aman\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [15]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())


2.11.0+cpu
CUDA available: False


## Creating Tensors

In [16]:
a = torch.tensor(5)
v = torch.tensor([1,2,3])
m = torch.tensor([[1,2],
                  [3,4]])
t = torch.rand(3,4,5)

In [17]:
print(a.shape)
print(a.dtype)
print(a.device)
print(a.ndim)

print(v.shape)
print(v.dtype)
print(v.device)
print(v.ndim)

print(m.shape)
print(m.ndim)

print(t.shape)
print(t.dtype)
print(t.ndim)


torch.Size([])
torch.int64
cpu
0
torch.Size([3])
torch.int64
cpu
1
torch.Size([2, 2])
2
torch.Size([3, 4, 5])
torch.float32
3


In [18]:
# From a list
t1 = torch.tensor([1, 2, 3])

# Zeros, ones, random
t2 = torch.zeros(3, 4)
t3 = torch.ones(2, 2)
t4 = torch.rand(3, 3)          # uniform [0,1)
t5 = torch.randn(3, 3)         # standard normal



print(t2.shape, t2.dtype)
print(t2)
print("t3 tesor\n", t3)
print("t5 tesor\n",t5)


torch.Size([3, 4]) torch.float32
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
t3 tesor
 tensor([[1., 1.],
        [1., 1.]])
t5 tesor
 tensor([[ 0.9958,  0.5388, -0.3576],
        [ 0.5973, -1.4141,  1.3249],
        [-0.0648, -0.4639,  1.4980]])


In [19]:
# From a range
t6 = torch.arange(0, 10, 2)
print("t6 tesor\n",t6)

t6 tesor
 tensor([0, 2, 4, 6, 8])


In [20]:
# NumPy
import numpy as np
arr = np.array([1, 2, 3])
t7 = torch.from_numpy(arr)

Key attributes: `.shape`, `.dtype`, `.device`, `.ndim`

## Basic Operations

In [21]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

print(a + b)          # element-wise add
print(a * b)          # element-wise multiply (Hadamard product)
print(a @ b)          # dot product (1-D tensors)
print(torch.dot(a, b))# same as above
print(a.sum(), a.mean(), a.max())


tensor([5., 7., 9.])
tensor([ 4., 10., 18.])
tensor(32.)
tensor(32.)
tensor(6.) tensor(2.) tensor(3.)


**Broadcasting** — smaller tensors are automatically "stretched" to match shapes:

In [22]:
m = torch.ones(3, 3)
v = torch.tensor([1., 2., 3.])
print(m + v)   # v is broadcast across each row


tensor([[2., 3., 4.],
        [2., 3., 4.],
        [2., 3., 4.]])


## Matrix Multiplication — The Heart of Deep Learning

Every linear/fully-connected layer computes `y = xW + b`. This is matrix multiplication at scale, repeated millions/billions of times during training. Three equivalent ways to do it in PyTorch:

In [23]:
A = torch.rand(4, 3)
B = torch.rand(3, 5)

C1 = torch.matmul(A, B)
C2 = A @ B              # preferred, Pythonic operator overload
C3 = torch.mm(A, B)     # 2-D only, slightly stricter

print(torch.allclose(C1, C2), torch.allclose(C2, C3))


True True


`torch.dot` only works for 1-D vectors; `matmul`/`@` generalizes to batched, N-D tensors — this is what makes it usable inside real neural network layers.

##  Why Are Tensor Operations Faster Than Python Loops?

A Python `for` loop:
- Executes **one scalar operation at a time**, each with Python interpreter overhead (type-checking, dynamic dispatch, object creation)
- Cannot exploit modern hardware parallelism

A tensor operation:
- Is implemented in **compiled C++/CUDA** under the hood
- Uses **SIMD (Single Instruction, Multiple Data)** vectorization on CPU, and **thousands of cores in parallel** on GPU
- Processes contiguous blocks of memory efficiently (cache-friendly)

This is called **vectorization** — replacing explicit loops with array/tensor-level operations. It's one of the most important practical lessons in scientific computing and deep learning engineering.

## Code Efficiency

In [24]:

def traditional_python(x, w, b):
    output = b
    for x_j, w_j in zip(x, w):
        output += x_j * w_j
    return output


In [25]:
import random
random.seed(49)

b = 0.
x = [random.random() for _ in range(1000)]
w = [random.random() for _ in range(1000)]

In [26]:
%timeit traditional_python(x, w, b)

45.7 μs ± 950 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [27]:
def pytorch_dot(x, w, b):
    return x.dot(w) + b

In [28]:
t_b = torch.tensor(b)
t_x = torch.tensor(x)
t_w = torch.tensor(w)

In [29]:
%timeit pytorch_dot(t_x, t_w, t_b)

5.14 μs ± 307 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


## GPU Acceleration (if available)

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

x = torch.rand(1000, 1000).to(device)
y = torch.rand(1000, 1000).to(device)
%timeit z = x @ y   # runs on GPU if available
print(z.shape)


Using: cpu
6.56 ms ± 73.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


NameError: name 'z' is not defined

## Autograd — Automatic Differentiation
#### Training a neural network means adjusting weights to minimize a loss function, and that requires computing the gradient of the loss with respect to every parameter (backpropagation).
#### Autograd is PyTorch's system that computes these gradients automatically, without you deriving or coding any calculus by hand.

In [ ]:
import torch
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b        # forward pass — graph is built silently
loss = (y - 10) ** 2  # some loss

loss.backward()       # backward pass — autograd computes it all

print(w.grad)  # dLoss/dw
print(x.grad)  # dLoss/dx
print(b.grad)  # dLoss/db


In [ ]:
# Forward pass (same as PyTorch version)
x = 2.0
w = 3.0
b = 1.0

y = w * x + b          # y = 3*2 + 1 = 7
loss = (y - 10) ** 2    # loss = (7-10)^2 = 9

# Backward pass — YOU derive and code the chain rule by hand
# loss = (y - 10)^2  →  d(loss)/dy = 2*(y - 10)
dloss_dy = 2 * (y - 10)

# y = w*x + b  →  dy/dw = x,  dy/dx = w,  dy/db = 1
dloss_dw = dloss_dy * x      # chain rule: dloss/dw = dloss/dy * dy/dw
dloss_dx = dloss_dy * w
dloss_db = dloss_dy * 1

print(dloss_dw)   # matches w.grad
print(dloss_dx)   # matches x.grad
print(dloss_db)   # matches b.grad